# 05 - Sessions, multi-turn conversations, and streaming

Everything so far was one-shot: one prompt, one run, context discarded. Real
applications need follow-ups, recovery after a crash, and output that appears as
it is produced rather than four minutes later.

**Docs:** [Sessions](https://code.claude.com/docs/en/agent-sdk/sessions) ·
[Streaming output](https://code.claude.com/docs/en/agent-sdk/streaming-output)

In [ ]:
# --- boot cell: run this first in every notebook -------------------------
import os, sys, json, asyncio
from pathlib import Path
from dotenv import load_dotenv

# The SDK reads ANTHROPIC_API_KEY from the process environment.
# It does NOT parse .env for you, so we do it here.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Copy .env.example to .env and add your key."
)

MODEL = os.environ.get("COURSE_MODEL") or None   # None = SDK default
print("key loaded, model =", MODEL or "(SDK default)")

In [ ]:
from claude_agent_sdk import (
    query, ClaudeSDKClient, ClaudeAgentOptions,
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
)

SCRATCH = ROOT / "notebooks" / "_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)
(SCRATCH / "orders.csv").write_text(
    "id,customer,total,status\n"
    "A-1001,Rao,82.50,delivered\n"
    "A-1002,Webb,19.99,in_transit\n"
    "A-1003,Nair,240.00,cancelled\n",
    encoding="utf-8",
)

def show(message, prefix="  "):
    if isinstance(message, AssistantMessage):
        for b in message.content:
            if isinstance(b, TextBlock) and b.text.strip():
                print(f"{prefix}{b.text.strip()[:200]}")
            elif isinstance(b, ToolUseBlock):
                print(f"{prefix}[{b.name}]")
    elif isinstance(message, ResultMessage):
        print(f"{prefix}({message.subtype}, ${message.total_cost_usd or 0:.4f})")

print("ready")

## 5.1 What a session is

A session is the conversation history the SDK accumulates: your prompts, every
tool call, every tool result, every response. The SDK writes it to disk
automatically at `~/.claude/projects/<encoded-cwd>/<session-id>.jsonl`.

Returning to a session means the agent has everything from before — files it
read, analysis it did, decisions it made — without paying to re-derive them.

Four ways to handle sessions:

| Your app | Use |
|---|---|
| One prompt, no follow-up | nothing. One `query()` call |
| Multi-turn chat in one process | **`ClaudeSDKClient`** |
| Resume after a process restart | `continue_conversation=True` |
| Resume a *specific* past session | capture the id, pass `resume=...` |
| Branch without losing the original | `resume=..., fork_session=True` |

## 5.2 `ClaudeSDKClient`: multi-turn without ID juggling

The client holds the session for you. Each `client.query()` continues the same
conversation. Use it as an async context manager.

Note `receive_response()` — it iterates the messages for the **current** query
and stops at its result, which is what makes a turn-by-turn loop readable.

In [ ]:
opts = ClaudeAgentOptions(
    allowed_tools=["Read", "Glob", "Grep"],
    tools=["Read", "Glob", "Grep"],
    permission_mode="dontAsk",
    cwd=str(SCRATCH), setting_sources=[], max_turns=8, model=MODEL,
)

async with ClaudeSDKClient(options=opts) as client:
    await client.query("Read orders.csv. How many orders are there?")
    async for m in client.receive_response():
        show(m)

    print("\n--- follow-up, no ID passed anywhere ---")
    # "those" only resolves because the session kept the first exchange.
    await client.query("Which of those has the highest total?")
    async for m in client.receive_response():
        show(m)

The second question said "those" and it worked. The file was read once,
not twice.

`ClaudeSDKClient` also gives you things `query()` cannot do:

```python
await client.interrupt()                        # stop a running turn
await client.set_permission_mode("acceptEdits") # loosen mid-session
await client.set_model("claude-haiku-4-5")      # switch models mid-session
await client.get_server_info()                  # inspect MCP status
```

## 5.3 Capturing and resuming a session ID

For anything that outlives one process — a web request, a queue worker, a
retried CI job — you need the ID.

Read it from `ResultMessage.session_id`. It is present on **every** result,
success or error, which is what makes "resume after hitting the turn limit"
work.

In [ ]:
session_id = None

try:
    async for m in query(
        prompt="Read orders.csv and summarise it in one sentence.",
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Glob"], tools=["Read", "Glob"],
            permission_mode="dontAsk", cwd=str(SCRATCH),
            setting_sources=[], max_turns=6, model=MODEL,
        ),
    ):
        if isinstance(m, ResultMessage):
            session_id = m.session_id
        show(m)
except Exception as exc:
    print("raised:", str(exc)[:80])

print("\nsession_id:", session_id)

In [ ]:
# A completely separate query() call - it still has the earlier context.
async for m in query(
    prompt="Now list only the cancelled ones. Do not re-read the file.",
    options=ClaudeAgentOptions(
        resume=session_id,                       # <-- the whole mechanism
        allowed_tools=["Read", "Glob"], tools=["Read", "Glob"],
        permission_mode="dontAsk", cwd=str(SCRATCH),
        setting_sources=[], max_turns=6, model=MODEL,
    ),
):
    show(m)

### Recovering from a turn limit

The most practical use of resume. The run hit `error_max_turns`, but the id
survived, so you continue instead of starting over — and you keep everything it
already worked out.

```python
if result.subtype == "error_max_turns":
    async for m in query(
        prompt="Continue where you left off.",
        options=ClaudeAgentOptions(resume=result.session_id, max_turns=30),
    ):
        ...
```

## 5.4 `continue_conversation`: resume without tracking an ID

Picks up the most recent session in the current directory.

```python
options=ClaudeAgentOptions(continue_conversation=True)
```

Good for a CLI or a single-conversation desktop app. Wrong for a multi-user
service — "most recent in this directory" is not per-user isolation. There, hold
the ID per user and pass `resume`.

## 5.5 Forking: branch without losing the original

`resume=id` **plus** `fork_session=True` creates a new session seeded with a
copy of the original's history. The original is untouched. You end up with two
independent sessions.

Use it to try an alternative approach while keeping the option to go back.

> Forking branches the **conversation**, not the filesystem. If a forked agent
> edits files, those edits are real and visible to every session in that
> directory. To branch and revert file changes, use
> [file checkpointing](https://code.claude.com/docs/en/agent-sdk/file-checkpointing).

In [ ]:
forked_id = None

try:
    async for m in query(
        prompt="Suppose we archive cancelled orders instead of deleting them. "
               "What would change? Two sentences.",
        options=ClaudeAgentOptions(
            resume=session_id, fork_session=True,
            tools=[], setting_sources=[], max_turns=3, model=MODEL,
        ),
    ):
        if isinstance(m, ResultMessage):
            forked_id = m.session_id
        show(m)
except Exception as exc:
    print("raised:", str(exc)[:80])

print(f"\noriginal: {session_id}")
print(f"fork    : {forked_id}")
print("different:", session_id != forked_id)

## 5.6 Listing sessions

Both SDKs expose session inspection, useful for building a session picker, a
transcript viewer, or cleanup logic.

In [ ]:
from claude_agent_sdk import list_sessions, get_session_messages

sessions = list_sessions(directory=str(SCRATCH), limit=5)
for s in sessions:
    print(f"  {s.session_id[:8]}  {(s.summary or s.first_prompt or '')[:60]}")

if sessions:
    msgs = get_session_messages(sessions[0].session_id, directory=str(SCRATCH), limit=4)
    print(f"\n  first {len(msgs)} messages of {sessions[0].session_id[:8]}:")
    for m in msgs:
        print(f"    {m.type}")

Also available: `get_session_info()`, `rename_session()`,
`tag_session()`.

**Across machines.** Session files are local. To resume on a different host —
CI workers, ephemeral containers, serverless — either pass a
[`session_store` adapter](https://code.claude.com/docs/en/agent-sdk/session-storage)
so the SDK mirrors transcripts to your own backend, or move the `.jsonl` file.

Often the most robust option is neither: capture the *results* you need as
application state and put them in a fresh session's prompt. Shipping transcript
files around is more fragile than it looks.

## 5.7 Streaming output

By default you see a message once a turn completes. For a UI you want tokens as
they arrive. Set `include_partial_messages=True` and handle `StreamEvent`.

In [ ]:
from claude_agent_sdk import StreamEvent

async for m in query(
    prompt="Explain in about 60 words why agents need turn limits.",
    options=ClaudeAgentOptions(
        tools=[], include_partial_messages=True,
        setting_sources=[], max_turns=1, model=MODEL,
    ),
):
    if isinstance(m, StreamEvent):
        ev = m.event
        if ev.get("type") == "content_block_delta":
            delta = ev.get("delta", {})
            if delta.get("type") == "text_delta":
                print(delta["text"], end="", flush=True)
    elif isinstance(m, ResultMessage):
        print(f"\n\n({m.subtype}, ${m.total_cost_usd or 0:.5f})")

### Streaming *input* is a different thing

Two things share the word "streaming":

- **Streaming output** — `include_partial_messages=True`. Token deltas out.
- **Streaming input** — pass an async iterable as `prompt`. One `query()` call
  carries multiple user turns, each emitting its own `ResultMessage`. This is the
  mode for a live chat backend, and it changes how you read costs (see notebook
  08).

**Docs:** [Streaming vs single mode](https://code.claude.com/docs/en/agent-sdk/streaming-vs-single-mode)

## 5.8 Context and compaction

Context does **not** reset between turns. It accumulates: system prompt, tool
definitions, every prompt, every tool input, every tool output.

When the window approaches its limit the SDK **compacts** automatically —
summarises older history, keeps recent exchanges — and emits a `SystemMessage`
with `subtype="compact_boundary"`.

The consequence people get bitten by: **instructions from early in a
conversation may not survive compaction.** A rule you stated in your first
prompt can be summarised away by turn 40.

Where to put persistent rules instead:

- the **system prompt** — re-sent on every request
- **`CLAUDE.md`** loaded via `setting_sources` — re-injected every request
- a **`UserPromptSubmit` hook** that re-injects context (notebook 06)

Four ways to keep context small:

1. **Subagents** for subtasks — the parent only sees the summary (notebook 07)
2. **Fewer tools per agent** — every schema costs context
3. **Summarise inside tool handlers** — do not return 4,000 lines of JSON
4. **Lower `effort`** for routine work

## 5.9 Exercises

1. Build a 3-turn conversation with `ClaudeSDKClient` where each turn depends on
   the last. Print the cost per turn. Does it rise? Why?
2. Run a task with `max_turns=2` so it fails, then resume it with `max_turns=20`
   and finish the job.
3. Fork a session twice, ask each fork a different question, then resume the
   original. Confirm all three are independent.
4. Add a `PreCompact` hook (notebook 06 covers hooks) that prints when
   compaction fires. Then run a long enough session to trigger it.

In [ ]:
# Scratch space for the exercises.

## Next

[**06 - Hooks, permissions, and guardrails**](06_hooks_and_guardrails.ipynb) —
running your own code inside the agent loop, and making rules that the prompt
cannot talk its way past.

---

### References

- [Work with sessions](https://code.claude.com/docs/en/agent-sdk/sessions)
- [Session storage adapters](https://code.claude.com/docs/en/agent-sdk/session-storage)
- [Stream responses in real time](https://code.claude.com/docs/en/agent-sdk/streaming-output)
- [Streaming vs single mode](https://code.claude.com/docs/en/agent-sdk/streaming-vs-single-mode)
- [File checkpointing](https://code.claude.com/docs/en/agent-sdk/file-checkpointing)